# rank0-only-side-effects — ex2: rank-0 download + barrier so every rank reads safely

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `rank0-only-side-effects`. Running the final beacon cell reports progress against the `Distributed: rank-0-only side effects` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: rank-0-only side effects` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`rank0-only-side-effects`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "rank0-only-side-effects"
DD_SUBTOPIC = "Distributed: rank-0-only side effects"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Rank-0 side effects with `dist.barrier()` ordering — quick refresher

Pure `if rank == 0:` is enough for *fire-and-forget* writes (logs, checkpoints other ranks never read). When rank 0 produces something OTHER ranks must read (downloaded dataset, generated split file, tokenizer cache), you need a barrier so the readers don't race the writer:

```python
if rank == 0:
    download_dataset(url, target)     # rank-0-only side effect
dist.barrier()                        # everyone waits here
data = read_dataset(target)           # every rank can now read safely
```

**Why barrier on every rank.** `dist.barrier()` is a collective — every rank must call it, or the call deadlocks. Rank > 0 hits the barrier immediately and blocks; rank 0 does the download first, THEN hits the barrier, which unblocks everyone.

**Order matters at one place only.** The barrier goes BETWEEN the rank-0 side effect and the all-rank read. Pre-barrier, ranks > 0 wait. Post-barrier, the produced file is guaranteed to exist for every rank.

### Exercise 2 — rank-0 download + barrier so every rank reads safely

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Analyze
> LO: Analyze the writer-readers ordering pattern by combining the `if rank == 0:` guard for a download with a `dist.barrier()` so rank > 0's read is guaranteed to happen AFTER rank 0's write.
> Keywords: rank-0, barrier, ordering, download, shared-resource
> ```

**KCs targeted:** `rank0-side-effect-with-barrier`, `barrier-orders-writer-before-readers`

Implement `ex2_rank0_download_then_all_read(rank, world_size, dist_module, downloader, reader, log)`. The producer-consumer pattern that EVERY distributed training script needs once at startup:

1. **Rank 0 only — download.** If `rank == 0`, call `downloader()` and then `log('rank0-downloaded')` so the test can verify the order.
2. **Every rank — barrier.** Call `dist_module.barrier()` unconditionally. Ranks > 0 hit this first and block; rank 0 hits it after step 1 and unblocks everyone.
3. **Every rank — read.** Call `value = reader()` (the file is now guaranteed to exist on shared storage), then `log(f'rank{rank}-read-{value}')`. Return `value`.

Critical ordering invariants the test checks:
- `rank0-downloaded` appears in the log BEFORE any `rank*-read-...` entry.
- `downloader()` is called EXACTLY ONCE across all ranks (only rank 0 invokes it).
- `reader()` is called once per rank (every rank reads).

Input: `rank`, `world_size` ints; `dist_module` (mocked dist); `downloader`, `reader`, `log` callables.
Output: the value returned by `reader()` — same on every rank (reader returns a constant in the mock).

In [ ]:
def ex2_rank0_download_then_all_read(rank, world_size, dist_module,
                                    downloader, reader, log):
    if rank == 0:
        downloader()
        log('rank0-downloaded')
    dist_module.barrier()
    value = reader()
    log(f'rank{rank}-read-{value}')
    return value


<details><summary>Solution</summary>

```python
def ex2_rank0_download_then_all_read(rank, world_size, dist_module,
                                    downloader, reader, log):
    if rank == 0:
        downloader()
        log('rank0-downloaded')
    dist_module.barrier()
    value = reader()
    log(f'rank{rank}-read-{value}')
    return value
```

**Without the barrier, rank 1 races.** A bare `if rank == 0:` guarded write returns immediately on rank 0; meanwhile rank 1 tries to call `reader()` before rank 0 has finished writing. The file may be missing, half-written, or — worst — exist but with stale contents from a previous run. The barrier is the one line that prevents the race.

**Every rank must call barrier.** A barrier on rank 0 alone doesn't help — barrier is a collective. If rank 1 skips it, rank 0 hangs forever waiting for the world to catch up.

**Real DDP recipe:** rank 0 calls `download + write to NFS / S3`, `dist.barrier()`, every rank reads from the shared store. Same shape as this drill — just bigger downloader.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()